# Qwen Voice Agent - Google Colab Version

This notebook runs a voice-enabled AI assistant using:
- **Speech-to-Text**: Faster Whisper
- **Language Model**: Qwen3 1.7B (via Ollama)
- **Text-to-Speech**: Qwen3-TTS-12Hz-0.6B

⚠️ **Important**: Make sure to enable GPU in Runtime → Change runtime type → T4 GPU

## 1. Install Dependencies

In [ ]:
# # Install uv package manager
!pip install uv

In [ ]:
# Install required packages
!uv pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!uv pip install -q faster-whisper
!uv pip install -q langchain-core langchain-ollama
!uv pip install -q soundfile
!uv pip install -q transformers accelerate
!uv pip install -q ipywidgets
!uv pip install -q flash-attn --no-build-isolation

# Install Qwen-TTS
!uv pip install -q git+https://github.com/QwenLM/Qwen-TTS.git

print("✅ All dependencies installed!")

## 2. Install and Setup Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in background
import subprocess
import time

# Start Ollama server
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Wait for server to start

# Pull the Qwen3 1.7B model
!ollama pull qwen3:1.7b

print("✅ Ollama setup complete!")

## 3. Import Libraries and Setup

In [ ]:
import os
import torch
import soundfile as sf
import IPython.display as ipd
from faster_whisper import WhisperModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from qwen_tts import Qwen3TTSModel
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

# Create directories
os.makedirs('audios', exist_ok=True)

# Check GPU availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🔥 Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not available'}")

## 4. Configuration

In [ ]:
# Voice configurations for TTS
VOICE_CONFIG = {
    "English (US) - Ryan": {
        "language": "English",
        "speaker": "Ryan",
        "instruct": "Speak with enthusiasm and clarity"
    },
    "English (US) - Aiden": {
        "language": "English",
        "speaker": "Aiden",
        "instruct": "Speak naturally with a clear voice"
    },
    "Indonesian": {
        "language": "English",  # Qwen3-TTS doesn't have Indonesian, fallback to English
        "speaker": "Ryan",
        "instruct": "Speak clearly and naturally"
    }
}

# Language Model (LLM) configurations - separate from TTS
LLM_MODELS = {
    "Qwen 3 1.7B": "qwen3:1.7b",
}

# Assistant template
assistant_template = """
You are a helpful, conversational assistant. Keep replies short and clear.
User: {input}
Assistant:
"""

# Global variables for models
stt_model = None
tts_model = None
chat_history = []

## 5. Load Models

In [ ]:
def load_stt():
    """Load Speech-to-Text model (Whisper)"""
    global stt_model
    if stt_model is None:
        print("Loading Whisper STT model...")
        stt_model = WhisperModel("base", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
        print("✅ Whisper STT model loaded!")
    return stt_model

def load_tts():
    """Load Text-to-Speech model (Qwen3-TTS)"""
    global tts_model
    if tts_model is None:
        print("Loading Qwen3-TTS model...")
        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        attn = "flash_attention_2" if torch.cuda.is_available() else "eager"
        
        tts_model = Qwen3TTSModel.from_pretrained(
            "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
            device_map=device,
            dtype=dtype,
            attn_implementation=attn,
        )
        print("✅ Qwen3-TTS model loaded!")
    return tts_model

# Load models
stt = load_stt()
tts = load_tts()

## 6. Core Functions

In [ ]:
def transcribe_audio(file_path):
    """Transcribe audio using Whisper"""
    stt_model = load_stt()
    segments, _ = stt_model.transcribe(file_path)
    return " ".join([segment.text for segment in segments])

def generate_response(text, model_name):
    """Generate LLM response using Qwen via Ollama"""
    try:
        model = ChatOllama(model=model_name)
        prompt = ChatPromptTemplate.from_template(assistant_template)
        chain = prompt | model
        response = chain.invoke({"input": text})
        return response.content
    except Exception as e:
        if "ConnectError" in str(type(e).__name__):
            raise ConnectionError("Ollama is not running. Start it with: ollama serve")
        raise

def synthesize_audio(text, output_path, voice_config):
    """Synthesize audio using Qwen3-TTS"""
    tts_model = load_tts()
    
    wavs, sr = tts_model.generate_custom_voice(
        text=text,
        language=voice_config["language"],
        speaker=voice_config["speaker"],
        instruct=voice_config["instruct"]
    )
    
    # Save to file
    sf.write(output_path, wavs[0], sr)
    return output_path, sr

## 7. Interactive UI

In [ ]:
# Settings widgets
llm_dropdown = widgets.Dropdown(
    options=list(LLM_MODELS.keys()),
    value=list(LLM_MODELS.keys())[0],
    description='LLM Model:',
    style={'description_width': 'initial'}
)

voice_dropdown = widgets.Dropdown(
    options=list(VOICE_CONFIG.keys()),
    value=list(VOICE_CONFIG.keys())[0],
    description='TTS Voice:',
    style={'description_width': 'initial'}
)

upload_button = widgets.FileUpload(
    accept='.wav,.mp3,.ogg,.flac',
    multiple=False,
    description='Upload Audio'
)

process_button = widgets.Button(
    description='🎙️ Process Audio',
    button_style='success',
    icon='microphone'
)

clear_button = widgets.Button(
    description='🗑️ Clear History',
    button_style='warning',
    icon='trash'
)

output_area = widgets.Output()

def process_audio(b):
    with output_area:
        clear_output(wait=True)
        
        if not upload_button.value:
            print("⚠️ Please upload an audio file first!")
            return
        
        # Save uploaded audio
        uploaded_file = list(upload_button.value.values())[0]
        audio_path = 'audios/input.wav'
        with open(audio_path, 'wb') as f:
            f.write(uploaded_file['content'])
        
        print("🎧 Transcribing audio...")
        transcription = transcribe_audio(audio_path)
        
        # Add to chat history
        chat_history.append({"role": "user", "content": transcription})
        
        print("\n👤 User:")
        print(transcription)
        
        # Generate response
        print("\n💻 Generating response with LLM...")
        llm_model = LLM_MODELS[llm_dropdown.value]
        response = generate_response(transcription, llm_model)
        
        print("\n🤖 Assistant:")
        print(response)
        
        # Synthesize audio
        print("\n🔊 Synthesizing speech with Qwen3-TTS...")
        voice_config = VOICE_CONFIG[voice_dropdown.value]
        response_path = f'audios/response_{len(chat_history)}.wav'
        synthesize_audio(response, response_path, voice_config)
        
        # Add to chat history
        chat_history.append({
            "role": "assistant", 
            "content": response,
            "audio_path": response_path
        })
        
        # Play audio
        print("\n🔊 Audio Response:")
        display(Audio(response_path, autoplay=True))
        
        print("\n" + "="*50)
        print(f"📊 Total messages: {len(chat_history)}")

def clear_history(b):
    global chat_history
    chat_history = []
    with output_area:
        clear_output()
        print("✅ Chat history cleared!")

process_button.on_click(process_audio)
clear_button.on_click(clear_history)

# Display UI
print("🎙️ Qwen Voice Agent - Colab Edition")
print("="*50)
print("\n⚙️ Settings:")
display(llm_dropdown)
display(voice_dropdown)
print("\n📤 Upload Audio File:")
display(upload_button)
print("\n🎬 Actions:")
display(widgets.HBox([process_button, clear_button]))
print("\n📋 Output:")
display(output_area)

# Display current settings
with output_area:
    print(f"✅ Ready! Current settings:")
    print(f"  - LLM Model: {llm_dropdown.value} ({LLM_MODELS[llm_dropdown.value]})")
    print(f"  - TTS Voice: {voice_dropdown.value}")
    print(f"  - Device: {device}")
    print(f"\n👆 Upload an audio file and click 'Process Audio' to start!")

## 8. Alternative: Text-Only Chat (No Audio Upload Needed)

In [ ]:
# Text input for quick testing
text_input = widgets.Textarea(
    placeholder='Type your message here...',
    description='Message:',
    layout=widgets.Layout(width='100%', height='80px')
)

send_button = widgets.Button(
    description='📤 Send',
    button_style='info',
    icon='paper-plane'
)

text_output = widgets.Output()

def send_message():
    with text_output:
        clear_output(wait=True)
        
        if not text_input.value.strip():
            print("⚠️ Please enter a message!")
            return
        
        user_text = text_input.value
        text_input.value = ''  # Clear input
        
        print("👤 User:")
        print(user_text)
        
        # Generate response
        print("\n💻 Generating response...")
        llm_model = LLM_MODELS[llm_dropdown.value]
        response = generate_response(user_text, llm_model)
        
        print("\n🤖 Assistant:")
        print(response)
        
        # Synthesize audio
        print("\n🔊 Synthesizing speech...")
        voice_config = VOICE_CONFIG[voice_dropdown.value]
        response_path = f'audios/text_response_{len(chat_history)}.wav'
        synthesize_audio(response, response_path, voice_config)
        
        # Play audio
        print("\n🔊 Audio Response:")
        display(Audio(response_path, autoplay=True))
        
        print("\n" + "="*50)

send_button.on_click(send_message)

print("\n💬 Text Chat Mode (Alternative)")
print("="*50)
display(text_input)
display(send_button)
display(text_output)

## 9. View Chat History

In [ ]:
def display_chat_history():
    print("📜 Chat History")
    print("="*50)
    
    if not chat_history:
        print("No messages yet.")
        return
    
    for i, msg in enumerate(chat_history, 1):
        icon = "👤" if msg["role"] == "user" else "🤖"
        print(f"\n{icon} {msg['role'].upper()} (#{i}):")
        print(msg["content"])
        
        if "audio_path" in msg and os.path.exists(msg["audio_path"]):
            print("🔊 Audio:")
            display(Audio(msg["audio_path"]))
        
        print("-" * 50)

display_chat_history()